# KAMP CNC 금속가공 예시 — S1: 문제 재정의

## 학습 목표
- 현장 문제를 AI 과제로 전환하는 방법 실습
- 5 Whys 기법으로 근본 원인 분석
- KAMP 데이터를 활용한 AI 적합성 판단

## 시나리오
**충청남도 A 금속가공 회사** — CNC 선반 5대 운영
- 증상: 불량률 8%, 공구 파손으로 월 2회 라인 정지
- 목표: AI 에이전틱 봇으로 예지보전 + 품질 최적화

## KAMP 데이터 개요

**KAMP(Korea AI Manufacturing Platform)**는 한국스마트제조산업협회(KOSMO)가 운영하는 제조 AI 공개 데이터 플랫폼입니다.
금속가공·CNC 분야의 공정 데이터, 품질 데이터, 설비 상태 데이터를 무료로 제공합니다.

### CNC 금속가공 주요 데이터 컬럼

| 컬럼명 | 설명 | 단위 |
|--------|------|------|
| `timestamp` | 배치 완료 시점 타임스탬프 | datetime |
| `machine_id` | CNC 설비 ID | string |
| `spindle_rpm` | 주축 회전수 | RPM |
| `feed_rate_mm_min` | 이송 속도 | mm/min |
| `cutting_temp_c` | 절삭 온도 | °C |
| `vibration_x_um` | X축 진동 | μm |
| `vibration_y_um` | Y축 진동 | μm |
| `vibration_z_um` | Z축 진동 | μm |
| `current_a` | 주축 전류 | A |
| `tool_wear_index` | 공구 마모 지수 | 0~1 |
| `dimension_error_mm` | 치수 오차 | mm |
| `surface_roughness_ra` | 표면 거칠기 Ra | μm |
| `quality_pass` | 품질 합격 여부 | 0/1 |

> 📌 **KAMP 데이터 포털**: https://www.kamp-ai.kr — 회원가입 후 다운로드 가능

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# 90일치 데이터, 3대 CNC (CNC-01, CNC-02, CNC-03)
n_days = 90
records_per_day = 16  # 1.5시간마다 1배치

data = []
for machine_id in ['CNC-01', 'CNC-02', 'CNC-03']:
    for day in range(n_days):
        for rec in range(records_per_day):
            timestamp = pd.Timestamp('2025-10-01') + pd.Timedelta(days=day, hours=rec*1.5)
            
            # 공구 마모 패턴
            if machine_id == 'CNC-01':
                base_wear = 0.15 + (day / n_days) * 0.2  # 안정
            elif machine_id == 'CNC-02':
                base_wear = 0.20 + (day / n_days) * 0.65  # 위험 (현재 높음)
            else:  # CNC-03
                if day < 60:
                    base_wear = 0.10 + (day / 60) * 0.7
                else:
                    base_wear = 0.10 + ((day - 60) / 30) * 0.15  # 정비 완료 후 개선
            
            tool_wear = min(1.0, base_wear + np.random.normal(0, 0.03))
            
            # 파라미터
            spindle_rpm = max(500, 2000 - tool_wear * 300 + np.random.normal(0, 50))
            feed_rate = max(50, 200 - tool_wear * 80 + np.random.normal(0, 10))
            cutting_temp = 45 + tool_wear * 120 + np.random.normal(0, 5)
            vibration_x = 2.0 + tool_wear * 8.0 + np.random.normal(0, 0.5)
            vibration_y = 1.5 + tool_wear * 6.0 + np.random.normal(0, 0.4)
            vibration_z = 3.0 + tool_wear * 10.0 + np.random.normal(0, 0.6)
            current_a = 8.0 + tool_wear * 12.0 + np.random.normal(0, 1.0)
            
            # 품질
            defect_prob = 0.02 + tool_wear ** 2 * 0.5
            quality_pass = 1 if np.random.random() > defect_prob else 0
            dim_error = np.random.normal(0, 0.005 + tool_wear * 0.02)
            roughness = 0.8 + tool_wear * 2.5 + np.random.normal(0, 0.2)
            
            data.append({
                'timestamp': timestamp,
                'machine_id': machine_id,
                'spindle_rpm': round(spindle_rpm, 0),
                'feed_rate_mm_min': round(feed_rate, 1),
                'cutting_temp_c': round(cutting_temp, 1),
                'vibration_x_um': round(vibration_x, 2),
                'vibration_y_um': round(vibration_y, 2),
                'vibration_z_um': round(vibration_z, 2),
                'current_a': round(current_a, 2),
                'tool_wear_index': round(tool_wear, 3),
                'dimension_error_mm': round(dim_error, 4),
                'surface_roughness_ra': round(roughness, 2),
                'quality_pass': quality_pass,
            })

df = pd.DataFrame(data)
print(f"데이터 크기: {df.shape}")
print(f"기간: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"\n기계별 불량률:")
print(df.groupby('machine_id')['quality_pass'].apply(lambda x: f"{(1-x.mean())*100:.1f}%"))
df.head(3)

## 5 Whys 분석

**문제**: CNC-02 불량률 12%, 월 2회 공구 파손

| 단계 | 질문 | 답변 |
|------|------|------|
| Why 1 | 왜 불량이 발생하는가? | 공구 마모가 설계 수명보다 빨리 진행됨 |
| Why 2 | 왜 공구 마모가 빠른가? | 절삭 온도가 비정상적으로 높음 (165°C, 정상 120°C) |
| Why 3 | 왜 절삭 온도가 높은가? | 공구 마모 감지 없이 파라미터 최적화 안됨 |
| Why 4 | 왜 파라미터 최적화가 안 되는가? | 센서 데이터를 실시간으로 분석하는 시스템이 없음 |
| **Why 5 (근본 원인)** | 왜 실시간 분석 시스템이 없는가? | **설비 상태 기반 자동 의사결정 시스템 부재** |

---

### ✅ AI 과제 정의

> **예지보전 에이전트** — 공구 상태 실시간 모니터링 + 자동 경보 + 파라미터 최적화 권고

- **입력**: 센서 데이터 (온도, 진동, 전류, 공구 마모 지수)
- **출력**: 공구 교체 시점 예측 (잔여 수명) + 최적 절삭 파라미터 권고
- **에이전트 방식**: ReAct 루프 — 관측 → 분석 → 행동 (알람 발송 / 파라미터 조정 권고)

In [ ]:
criteria = {
    '기준': ['데이터 가용성', '패턴 반복성', 'ROI 명확성', '실시간성', 'AI 전문성 필요'],
    '점수 (1-5)': [5, 4, 5, 4, 3],
    '근거': [
        'KAMP 5,000행+, 15개 변수, 90일치',
        '공구 마모 → 온도 상승 → 불량 패턴 반복',
        '공구 파손 1회 = 50만원 손실, 라인 정지 4h',
        '배치 완료 시점 (1.5h 간격) 예측 충분',
        'ML 모델 + ReAct 에이전트 표준 패턴'
    ]
}
df_criteria = pd.DataFrame(criteria)
print("=== AI 적합성 평가 ===")
print(df_criteria.to_string(index=False))
total = sum(criteria['점수 (1-5)'])
print(f"\n총점: {total}/25 → {'✅ AI 적합' if total >= 18 else '⚠️ 재검토'}")

## Lab 1 과제: 현장 문제 재정의

### 지시사항
위 KAMP CNC 시나리오를 참고하여 본인 현장의 문제를 분석하세요.

| 항목 | 내용 |
|------|------|
| 현장 문제 | (작성) |
| 증상/지표 | (작성) |
| 5 Whys 근본 원인 | (작성) |
| AI 과제 정의 | (작성) |
| 데이터 가용성 | (작성) |
| 예상 ROI | (작성) |